## IMPORTS

In [1]:
import os
import pandas as pd
import re 


## IMPORT DATAFRAME

In [3]:
import os

notebook_dir = os.getcwd()
print(notebook_dir)
PDF_DIR = os.path.join(notebook_dir, "data", "pdfs")
df_main = pd.read_csv(os.path.join("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/lawrence_2026.csv"))
print(df_main.size)
print(df_main.shape)
print(df_main.head())
df_main.to_csv(os.path.join("UNCLEAN_lawrence_2026.csv"), index=False)
target_csv = pd.read_csv(
    os.path.join( "UNCLEAN_lawrence_2026.csv")
)

/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts
342963
(38107, 9)
   Incident #                 Date             Type  \
0    26025341  2026-06-01 23:53:00  NOISE ORDINANCE   
1    26025340  2026-06-01 23:52:00      SEX OFFENSE   
2    26025338  2026-06-01 23:29:00      EXTRA CHECK   
3    26025337  2026-06-01 23:16:00      911 HANG-UP   
4    26025336  2026-06-01 23:15:00          MV STOP   

                                      Location Arrested  Name  DOB  Charges  \
0                       ALLSTON ST & HOWARD ST       No   NaN  NaN      NaN   
1                             75 MANCHESTER ST       No   NaN  NaN      NaN   
2  SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST       No   NaN  NaN      NaN   
3                                 98 OSGOOD ST       No   NaN  NaN      NaN   
4                                 20 ISLAND ST       No   NaN  NaN      NaN   

            Disposition  
0  SERVICED - SEE NOTES  
1       REPORT REQUIRED  
2  SERVICED

## DROP ANY EMPTY ROWS

In [4]:
columns_to_check = ['Incident #', 'Date', 'Type', 'Location']

empty_rows_count = target_csv[columns_to_check].isna().all(axis=1).sum()
print(f"{empty_rows_count} empty rows have been dropped")

target_csv = target_csv.dropna(subset = columns_to_check)

0 empty rows have been dropped


### Clean rows that only had one letter, i.e when cells only had "A" or "B"

In [5]:
pattern = r'^[A-Za-z]$'
matching_rows = target_csv['Type'].str.match(pattern, na=False)
target_csv_cleaned = target_csv[~matching_rows]

num_dropped_rows = matching_rows.sum()
print(f"Number of rows dropped: {num_dropped_rows}")

Number of rows dropped: 0


In [6]:
target_csv_backup = target_csv_cleaned.copy()  # Backup 

### Check 'Type' column

In [7]:
pd.set_option('display.max_rows', None)

print(target_csv_backup["Type"].value_counts())

Type
MV STOP                                8558
EXTRA CHECK                            3996
NOISE ORDINANCE                        2872
TOW TRESPASS                           1568
ALARM BURGLARY                         1249
VIOLATION OF CITY ORDINANCE            1080
DISTURBANCE                            1071
CHECK WELL BEING                       1067
CONTINUED INVESTIGATION                 919
SUSPICIOUS PERSON                       880
MV BLOCKING                             869
ASSIST EMS                              858
209A SERVE                              758
TRESPASSING                             740
KEEP THE PEACE                          734
TOW REPO                                676
LOUD NOISE                              637
HIT & RUN MOTOR VEHICLE                 625
AUTO ACCIDENT NO PERSONAL INJURY        406
TOW OF M/V                              364
GENERAL SERVICE                         361
UNWANTED GUEST                          332
LARCENY/PAST               

### Getting rid of '\nLocation', '\n', '\n172' and '\nhttps' in Type column

In [8]:
pattern = r'\n(Location|172|http)$'

# count of rows that will be modified
num_modified_rows = target_csv_cleaned['Type'].str.contains(pattern, regex=True, na=False).sum()

target_csv_backup.loc[:, 'Type'] = target_csv_cleaned['Type'].str.replace(pattern, '', regex=True)

print(f"Number of rows modified: {num_modified_rows}")

Number of rows modified: 0


/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_67726/1808532403.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  num_modified_rows = target_csv_cleaned['Type'].str.contains(pattern, regex=True, na=False).sum()


In [9]:
target_csv_backup['Type'] = target_csv_backup['Type'].str.replace('\n', '', regex=False).str.strip()
pd.set_option('display.max_rows', None)
target_csv_backup['Type'] = target_csv_backup['Type'].str.replace('lawpd', '', regex=False).str.strip()


## Random Messy Entries

In [10]:
entries_to_remove = [
    'MV/BLOCKINGMAL DAMAGE',
    'PublicLogFIRE WORKSFIRE WORKS',
    'SUS PERS/MVPublicLog',
    'M/V STOPALARM/BURG',
    'PublicLogTOW OF M/VKEEP PEACEALARMSDISABLED MV',
    'ESCORTCK WELL BEING',
    'Location',
    '172'
]

# count how many entries match the bad strings
num_to_remove = target_csv_backup['Type'].isin(entries_to_remove).sum()
print(f"Entries matching bad strings to remove: {num_to_remove}")

# remove these entries
target_csv_backup = target_csv_backup[~target_csv_backup['Type'].isin(entries_to_remove)]

# drop any remaining empty or whitespace-only 'Type'
before_drop = len(target_csv_backup)
target_csv_backup = target_csv_backup[target_csv_backup['Type'].str.strip() != '']
after_drop = len(target_csv_backup)

print(f"Additional empty/whitespace rows dropped: {before_drop - after_drop}")

Entries matching bad strings to remove: 0
Additional empty/whitespace rows dropped: 0


In [11]:
target_csv_backup['Type'] = target_csv_backup['Type'].str.replace('NOISE ORDDIRT BIKE', '', regex=False).str.strip()
target_csv_backup = target_csv_backup[target_csv_backup['Type'].str.strip() != '']

### Check "Type" again, ensure nothing out of ordinary

In [12]:
pd.set_option('display.max_rows', None)

print(target_csv_backup["Type"].value_counts())

Type
MV STOP                                8558
EXTRA CHECK                            3996
NOISE ORDINANCE                        2872
TOW TRESPASS                           1568
ALARM BURGLARY                         1249
VIOLATION OF CITY ORDINANCE            1080
DISTURBANCE                            1071
CHECK WELL BEING                       1067
CONTINUED INVESTIGATION                 919
SUSPICIOUS PERSON                       880
MV BLOCKING                             869
ASSIST EMS                              858
209A SERVE                              758
TRESPASSING                             740
KEEP THE PEACE                          734
TOW REPO                                676
LOUD NOISE                              637
HIT & RUN MOTOR VEHICLE                 625
AUTO ACCIDENT NO PERSONAL INJURY        406
TOW OF M/V                              364
GENERAL SERVICE                         361
UNWANTED GUEST                          332
LARCENY/PAST               

### 'FALLS BRIDGE\null' => 'FALLS BRIDGE'

In [13]:
pattern = r'^FALLS BRIDGE\s*/\s*null$'
num_modified_rows = target_csv_backup['Location'].str.contains(pattern, regex=True, na=False).sum()

target_csv_backup['Location'] = target_csv_backup['Location'].replace(pattern, 'FALLS BRIDGE', regex=True)


print(f"Number of rows modified: {num_modified_rows}")

Number of rows modified: 0


### Function to clean  split "Location Prefix" i.e Prefix\Address, print to csv for review

In [14]:
target_csv_modified = target_csv_backup.copy()

# split "Location Prefix" and "Location"
def separate_location(value):
    if isinstance(value, str) and '/' in value:  # check if it's a string and contains '/'
        parts = value.split('/', 1)  # split at the first '/'
        return value, parts[0].strip(), parts[1].strip()  # (Original, Prefix, Address)
    else:
        return value, "", value.strip()  # if no '/', Prefix is empty, Location is full value

# apply function to create new columns
target_csv_modified[['Original Location', 'Location Prefix', 'Location']] = target_csv_modified['Location'].apply(lambda x: pd.Series(separate_location(x)))

# save the corrected data to a CSV for review
output_dir = os.path.join(notebook_dir, "data", "missing_dates_csv", "md_intermediate")
os.makedirs(output_dir, exist_ok=True)

csv_filename = os.path.join(output_dir, "md_location_separation.csv")
target_csv_modified[["Original Location", "Location Prefix", "Location"]].to_csv(csv_filename, index=False)
print(f"✅ CSV file saved as: {csv_filename}")

✅ CSV file saved as: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_intermediate/md_location_separation.csv


In [15]:
target_csv_backup[['Location Prefix', 'Location']] = target_csv_backup['Location'].apply(
    lambda x: pd.Series(separate_location(x)[1:])  # slice to return only the last two elements, not 'original location'
)
target_csv_backup = target_csv_backup[target_csv_backup["Location"] != 'null']

In [16]:
pd.set_option('display.max_rows', None)
print(target_csv_backup["Location"].value_counts())

Location
MCDONALD'S, 50 BROADWAY                                                 360
LPD, LAWRENCE POLICE, 90 LOWELL ST                                      269
WASHINGTON MILL, 240 CANAL ST                                           266
S UNION ST                                                              220
CVS, 266 BROADWAY                                                       200
BUCKLEY GARAGE, 99 AMESBURY ST                                          179
PACIFIC MILLS LOFTS, 300 CANAL ST                                       147
TRAIN STATION - MCGOVERN TRANSPORTATION CENTER, 211 MERRIMACK ST        147
550 BROADWAY                                                            137
MARKET BASKET, 700 ESSEX ST                                             132
SOUTH COMMON, MARKET & S UNION                                          129
PEMBERTON STATE PARK, CANAL ST                                          127
PAC10 LOFTS, 300 METHUEN ST                                             124
BRO

## Checkpoint 1

In [17]:
md_file_path = os.path.join(notebook_dir,'data', 'missing_dates_csv')
md_cp_dir = os.path.join(md_file_path, "md_checkpoints")
os.makedirs(md_cp_dir, exist_ok=True)

cp1_file_path = os.path.join(md_cp_dir, "md_checkpoint1.csv")
target_csv_backup.to_csv(cp1_file_path, index=False)
print(f"✅ CSV file saved as: {cp1_file_path}")

✅ CSV file saved as: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/md_checkpoint1.csv


## Drop duplicate incident amounts 

In [18]:
checkpoint1 = pd.read_csv(cp1_file_path)
print(checkpoint1.head())

   Incident #                 Date             Type  \
0    26025341  2026-06-01 23:53:00  NOISE ORDINANCE   
1    26025340  2026-06-01 23:52:00      SEX OFFENSE   
2    26025338  2026-06-01 23:29:00      EXTRA CHECK   
3    26025337  2026-06-01 23:16:00      911 HANG-UP   
4    26025336  2026-06-01 23:15:00          MV STOP   

                                      Location Arrested  Name  DOB  Charges  \
0                       ALLSTON ST & HOWARD ST       No   NaN  NaN      NaN   
1                             75 MANCHESTER ST       No   NaN  NaN      NaN   
2  SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST       No   NaN  NaN      NaN   
3                                 98 OSGOOD ST       No   NaN  NaN      NaN   
4                                 20 ISLAND ST       No   NaN  NaN      NaN   

            Disposition Location Prefix  
0  SERVICED - SEE NOTES             NaN  
1       REPORT REQUIRED             NaN  
2  SERVICED - SEE NOTES             NaN  
3  SERVICED - SEE NOTES   

In [19]:
total_rows = len(checkpoint1)
unique_incidents = checkpoint1["Incident #"].nunique()

print(f"Total rows: {total_rows}")
print(f"Unique Incident #: {unique_incidents}")

# Drop dupes and save to new df
checkpoint1_no_dupes = checkpoint1.drop_duplicates(subset="Incident #")

rows_dropped = total_rows - len(checkpoint1_no_dupes)
print(f"Rows dropped: {rows_dropped}")

Total rows: 38107
Unique Incident #: 26502
Rows dropped: 11605


## DROP URLS

In [20]:
# Define regex pattern to match URLs (both "http://" and "https://")
pattern = r'http[s]?://\S+'

# Count rows affected by the pattern
affected_rows = checkpoint1_no_dupes['Charges'].str.contains(pattern, regex=True, na=False).sum()

checkpoint1_no_dupes['Charges'] = checkpoint1_no_dupes['Charges'].str.replace(pattern, '', regex=True).str.strip()

print(f"✅ Number of rows affected: {affected_rows}")

AttributeError: Can only use .str accessor with string values!

## Drop URL with project 2026 ##


In [21]:
pattern = r'http[s]?://\S+'

# Convert Charges to a string-compatible dtype while preserving missing values
checkpoint1_no_dupes['Charges'] = checkpoint1_no_dupes['Charges'].astype('string')

# Count rows affected by the pattern
affected_rows = checkpoint1_no_dupes['Charges'].str.contains(
    pattern, regex=True, na=False
).sum()

# Remove URLs from Charges
checkpoint1_no_dupes['Charges'] = (
    checkpoint1_no_dupes['Charges']
    .str.replace(pattern, '', regex=True)
    .str.strip()
)

print(f"✅ Number of rows affected: {affected_rows}")

✅ Number of rows affected: 0


/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_67726/2678624496.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checkpoint1_no_dupes['Charges'] = checkpoint1_no_dupes['Charges'].astype('string')
/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_67726/2678624496.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checkpoint1_no_dupes['Charges'] = (


## Make CSV to review

In [22]:
df_charges = checkpoint1_no_dupes[checkpoint1_no_dupes['Charges'].notna() & checkpoint1_no_dupes['Charges'].str.strip().ne('')]

csv_name = os.path.join(md_file_path, 'md_intermediate', 'md_charges_only.csv')

df_charges.to_csv(csv_name, index=False)

print(f"✅ Filtered dataset saved to: {csv_name}")
print(f"Number of rows retained: {len(df_charges)} out of {len(target_csv_backup)}")

✅ Filtered dataset saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_intermediate/md_charges_only.csv
Number of rows retained: 0 out of 38107


In [23]:
charges_csv = pd.read_csv(csv_name)
print(charges_csv.head())

Empty DataFrame
Columns: [Incident #, Date, Type, Location, Arrested, Name, DOB, Charges, Disposition, Location Prefix]
Index: []


In [24]:
target_csv_backup2 = charges_csv.copy()

def split_multiple_arrests(row):
    """
    Splits multiple arrests stored in one row into multiple rows, 
    while keeping the first person in the original row.
    """
    incident_id = row["Incident #"]
    date = row["Date"]
    type_ = row["Type"]
    location = row["Location"]
    arrested = row["Arrested"]
    
    # First person (Original row should be preserved)
    original_name = row["Name"]
    original_dob = row["DOB"]
    original_charges = row["Charges"]

    # Identify additional names in the Charges column
    person_pattern = re.findall(
        r"Name:\s*([^,]+)\s*,\s*([^;]+)\s*Date of Birth:\s*([\d/]+);[^;]*;\s*(?:Charges|Warrant Charges)?:?\s*([^;]+)",
        original_charges
    )

    cleaned_rows = []
    
    # Preserve the first person in the original row (keep only their charges)
    cleaned_rows.append({
        "Incident #": incident_id,
        "Date": date,
        "Type": type_,
        "Location": location,
        "Arrested": arrested,
        "Name": original_name,
        "DOB": original_dob,
        "Charges": original_charges.split("Name:")[0].strip()  # Keep only the first person's charges
    })

    # Add new rows for additional people found
    for match in person_pattern:
        last_name = match[0].strip()
        first_name = match[1].strip()
        dob = match[2].strip()
        charges = match[3].strip()

        cleaned_rows.append({
            "Incident #": incident_id,
            "Date": date,
            "Type": type_,
            "Location": location,
            "Arrested": arrested,
            "Name": f"{first_name} {last_name}",
            "DOB": dob,
            "Charges": charges
        })

    return cleaned_rows

# Process each row and split when necessary
cleaned_data = []
for _, row in target_csv_backup2.iterrows():
    cleaned_data.extend(split_multiple_arrests(row))

cleaned_df = pd.DataFrame(cleaned_data)

csv_name = os.path.join(md_file_path, 'md_intermediate', 'md_charges_unnested.csv')
cleaned_df.to_csv(csv_name, index=False)

print(f"✅ Cleaning complete! Saved to: {csv_name}")

✅ Cleaning complete! Saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_intermediate/md_charges_unnested.csv


In [25]:
csv_name = os.path.join(md_file_path, 'md_intermediate', 'md_charges_unnested.csv')


### Merge OG DataFrame with Charges DataFrame

In [26]:
# Checkpoint 2: merge cleaned/unnested arrest data back into full dataset

target_csv_backup = checkpoint1_no_dupes.copy()

# Remove old person/charge columns before merging cleaned versions back in
target_csv_backup = target_csv_backup.drop(
    columns=["Charges", "Name", "DOB"],
    errors="ignore"
)

# Merge cleaned arrest data back into the full dataset
df_final = target_csv_backup.merge(
    cleaned_df,
    on=["Incident #", "Date", "Type", "Location", "Arrested"],
    how="left"
)

checkpoint2 = df_final.copy()
checkpoint2.to_csv("checkpoint2.csv", index=False)
print("✅ Checkpoint 2 created")
print(f"Rows: {len(checkpoint2)}")
print(f"Columns: {len(checkpoint2.columns)}")

KeyError: 'Incident #'

## New merge project 2026

In [34]:
# Preserve the existing data when there are no arrest details to unnest
if cleaned_df.empty:
    print("No arrest details to unnest. Skipping merge.")
    df_final = checkpoint1_no_dupes.copy()

else:
    # Original logic for older reports with arrest details
    target_csv_backup = checkpoint1_no_dupes.copy()

    target_csv_backup = target_csv_backup.drop(
        columns=["Charges", "Name", "DOB"],
        errors="ignore"
    )

    df_final = target_csv_backup.merge(
        cleaned_df,
        on=["Incident #", "Date", "Type", "Location", "Arrested"],
        how="left"
    )

checkpoint2 = df_final.copy()

print("Checkpoint 2 rows:", len(checkpoint2))
display(checkpoint2.head())

No arrest details to unnest. Skipping merge.
Checkpoint 2 rows: 26502


,Incident #,Date,Type,Location,Arrested,Name,DOB,Charges,Disposition,Location Prefix
0,26025341,2026-06-01 23:53:00,NOISE ORDINANCE,ALLSTON ST & HOWARD ST,No,NaN,NaN,<NA>,SERVICED - SEE NOTES,NaN
1,26025340,2026-06-01 23:52:00,SEX OFFENSE,75 MANCHESTER ST,No,NaN,NaN,<NA>,REPORT REQUIRED,NaN
2,26025338,2026-06-01 23:29:00,EXTRA CHECK,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST",No,NaN,NaN,<NA>,SERVICED - SEE NOTES,NaN
3,26025337,2026-06-01 23:16:00,911 HANG-UP,98 OSGOOD ST,No,NaN,NaN,<NA>,SERVICED - SEE NOTES,NaN
4,26025336,2026-06-01 23:15:00,MV STOP,20 ISLAND ST,No,NaN,NaN,<NA>,SERVICED - SEE NOTES,NaN


In [35]:

# Save checkpoint 2 in your data directory
checkpoint2_path = os.path.join( "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint2.csv")

checkpoint2.to_csv(checkpoint2_path, index=False)

print(f"✅ Checkpoint 2 saved to: {checkpoint2_path}")
print(f"Total rows: {len(checkpoint2)}")
print(f"Total columns: {len(checkpoint2.columns)}")

✅ Checkpoint 2 saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint2.csv
Total rows: 26502
Total columns: 10


In [32]:
csv_name = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint2.csv"

checkpoint2 = pd.read_csv(csv_name)
checkpoint2.head()

,Incident #,Date,Type,Location,Arrested,Location Prefix,Name,DOB,Charges
0,25026764.0,2025-07-01 00:12:08,THREATS,259 ANDOVER ST,No,NaN,NaN,NaN,NaN
1,25026765.0,2025-07-01 00:19:36,TOW/REPOSSED,454 HAMPSHIRE ST,No,NaN,NaN,NaN,NaN
2,25026766.0,2025-07-01 00:49:07,DISTURBANCE,155 HAVERHILL ST,No,NaN,NaN,NaN,NaN
3,25026767.0,2025-07-01 01:15:01,CK WELL BEING,621 COMMON ST,No,NaN,NaN,NaN,NaN
4,25026769.0,2025-07-01 01:29:06,EXTRA SURVEIL,425 MERRIMACK ST,No,NaN,NaN,NaN,NaN
